In [ ]:
import requests
from requests.packages.urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import csv
from tqdm import tqdm

In [ ]:
class TimeoutHttpAdapter(HTTPAdapter):
    def __init__(self, timeout=None, *args, **kwargs):
        self.timeout = timeout
        if "timeout" in kwargs:
            del kwargs["timeout"]
        super().__init__(*args, **kwargs)

    def send(self, *args, **kwargs):
        kwargs['timeout'] = self.timeout
        return super().send(*args, **kwargs)

In [ ]:
r=requests.Session()
retry_strategy = Retry(
            total=3,
            status_forcelist=[104, 429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET" "POST", "PUT", "DELETE", "OPTIONS", "TRACE"],
            backoff_factor=2
        )
r.headers["User-Agent"]='My User Agent 1.0'
r.mount('https://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))
r.mount('http://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))

In [ ]:
rows=[]
for pgno in tqdm(range(3,15)):
    req = r.get('https://www.vidhikarya.com/get-free-legal-advice/sex-crime-legal-advice?page='+str(pgno))
    soup = BeautifulSoup(req.content, 'html.parser')
    s = soup.find_all('h5', class_='card-title QuestionTitle')
    link_list=[]
    for li in s:
        a = li.find('a')
        if a:
            link_list.append('https://www.vidhikarya.com'+a.attrs["href"])
    for qlink in link_list:
        qreq=r.get(qlink)
        qsoup=BeautifulSoup(qreq.content, "html.parser")
        title=qsoup.find('h1',class_='card-title QuestionTitle').text.strip()
        question=qsoup.find('p',class_='card-text QuestionContent').text.strip().replace('\n',' ')
        answer=qsoup.find('span',class_='card-text comment text').text.strip().replace('\n',' ')
        rows.append([title,question,answer])


In [ ]:
fields = ['title', 'question','answer'] 
with open('', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)
    f.close()